# Trace a CrewAI trip-planning crew

You will build a two-agent [CrewAI](https://www.crewai.com/) crew — a Researcher who searches the
web and a Planner who turns that research into an itinerary — and run it twice in one conversation:
once to plan a trip, once to revise it. Every agent, tool and model call lands in AcruxCore
automatically.

**There is no AcruxCore code in the crew.** Not one line. That is the whole point of this notebook,
and it makes it the odd one out in this set: no prompts to version, no tool catalog, no gateway.
CrewAI runs its own agents, its own tool loop and its own model calls, and reports what it did over
[OpenTelemetry](https://opentelemetry.io/) — the protocol nearly every agent framework speaks.
AcruxCore accepts that protocol at `POST /api/v1/traces/otlp`, so pointing a working crew at us is
three lines of setup outside the crew.

| Piece | What it does | Who runs it |
|---|---|---|
| `register()` | builds a standard OTel pipeline aimed at our OTLP endpoint | **your code**, once at startup |
| `openinference-instrumentation-crewai` | records which agent ran, which task, which tool | a library, patching CrewAI |
| `openinference-instrumentation-openai` | records the model call: model, tokens, cost | a library, patching `openai` |
| the Researcher and the Planner | plan and revise the trip | **CrewAI**, knowing nothing about us |
| `using_session(...)` | ties both runs together as one conversation | **your code** |

Every cell runs against a real account, the real OpenAI API and the real Tavily API. Nothing here
is faked or mocked.

**Two ways to do every step.** The one step that changes something on the platform has two
headings: **In the dashboard**, with the values to set, and **The same thing in code**. They are not
two different features — the dashboard and the SDK call the same API. Pick either.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | changes something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Trace a CrewAI Trip-Planning Crew](https://docs.acruxcore.com/docs/tutorials/trace-a-crewai-trip-planner)

---

## Step 0 — What you need before you start

**1. A personal API key.** **Account & keys → New key**. Copy it the moment it appears — that is
the only time the full value is shown.

**2. An OpenAI API key.** The crew calls `gpt-4o-mini` directly, not through our gateway, so this
key goes to OpenAI and never to us.

Nothing here needs OpenAI in particular. CrewAI resolves the `llm=` string through LiteLLM, so
`llm="openrouter/google/gemini-3.7-flash"` with `OPENROUTER_API_KEY` set runs the same crew on
another provider — tested, and the only lines that change are this key and `MODEL`. The tracing
below is unaffected either way, because it watches the framework rather than the provider.

**3. A Tavily API key.** The Researcher searches with it. The free tier is enough.

**4. Python 3.10 to 3.13.** CrewAI does not support 3.14 yet. This notebook was run on 3.12; if
your kernel is newer, make a 3.12 environment for it first.

**5. Six packages.** Two are the framework, one is our OTel helper, two are the instrumentors that
do the actual recording, and the last one is easy to miss.

`crewai-tools` installs the *wrapper* for Tavily search but not the `tavily-python` client it calls.
Leave it out and the tool does not fail — it asks, on standard input, whether to install it for you.
In a notebook that prompt has nowhere to go, so the cell aborts halfway through building the crew.

In [ ]:
%pip install -q --upgrade crewai crewai-tools tavily-python "acruxcore[otel]" \
    openinference-instrumentation-crewai openinference-instrumentation-openai

**Setup.** Set the three keys.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your shell
before you start Jupyter, and treat this cell as a fallback.

The fourth variable is not ours. CrewAI 1.x has **its own** tracing service, unrelated to
AcruxCore, and it prints a panel asking about it on first run. Setting it to `false` keeps the
output clean. Turning it off has no effect on anything in this notebook.

In [1]:
import json
import os

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")
os.environ.setdefault("OPENAI_API_KEY", "sk-...")
os.environ.setdefault("TAVILY_API_KEY", "tvly-...")

# CrewAI's own tracing service, nothing to do with us. Off, so no prompt appears.
os.environ["CREWAI_TRACING_ENABLED"] = "false"

MODEL = "gpt-4o-mini"
SESSION_ID = "crewai-trip-planner-notebook"

# Do NOT print the base URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Four things, in the order they fail. Read the instrumentor versions even when they pass:
every span name in this notebook comes from those two libraries, and they change between releases.

In [2]:
import importlib.metadata as metadata

import requests

from acruxcore import AcruxCore

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

# 1. Does the AcruxCore key work?
await hub.prompts.list(limit=1)
print("acruxcore key: ok")

# 2. Is every piece installed, and which versions? Span names come from these.
for package in ("crewai", "crewai-tools", "acruxcore",
                "openinference-instrumentation-crewai",
                "openinference-instrumentation-openai", "opentelemetry-sdk"):
    print(f"  {package:<42} {metadata.version(package)}")

# 3. Does the OpenAI key work? The crew calls OpenAI directly, so this is not our problem to catch.
openai_probe = requests.post("https://api.openai.com/v1/chat/completions", timeout=60,
                             headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
                             json={"model": MODEL, "max_tokens": 1,
                                   "messages": [{"role": "user", "content": "hi"}]})
print("openai key:", "ok" if openai_probe.ok else f"FAILED {openai_probe.status_code}")

# 4. Does the Tavily key work?
tavily_probe = requests.post("https://api.tavily.com/search", timeout=60,
                            headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
                            json={"query": "test", "max_results": 1})
print("tavily key:", "ok" if tavily_probe.ok else f"FAILED {tavily_probe.status_code}")

acruxcore key: ok
  crewai                                     1.15.17
  crewai-tools                               1.15.17
  acruxcore                                  0.10.0
  openinference-instrumentation-crewai       1.1.12
  openinference-instrumentation-openai       0.1.54
  opentelemetry-sdk                          1.42.1
openai key: ok
tavily key: ok


---

## Step 1 — Who owns the agent loop

### The general problem

An observability tool can only describe work it knows about, and there are exactly two ways it can
find out.

It can **run the loop**. Your code asks the platform to call the model, so the platform sees every
request and response and writes the trace itself. That is what every other tutorial on this site
does, through the gateway or `run_tool_loop`.

Or the framework can **run its own loop and report afterwards**. CrewAI has its own agents, its own
task graph and its own tool dispatch. It is never going to hand that to somebody else's SDK. What
it does instead — like LangChain, LlamaIndex and the OpenAI Agents SDK — is emit OpenTelemetry
spans describing what it did.

The second shape is not a lesser version of the first. It is the only shape available once the
framework owns the loop.

### Where our case sits

`POST /api/v1/traces/otlp` accepts OTLP directly. `acruxcore.otel.register()` is a small helper
that builds the pipeline every OTel app needs — a `TracerProvider`, a `BatchSpanProcessor` and an
`OTLPSpanExporter` — pointed at `$ACRUXCORE_BASE_URL/traces/otlp`, using `$ACRUXCORE_API_KEY` as
the bearer token. It does nothing you could not write by hand in four lines.

The instrumentors are separate packages from the [OpenInference](https://github.com/Arize-ai/openinference)
project, not ours. They patch the framework and emit spans carrying OpenInference attributes.
Our OTLP endpoint reads those attributes and maps them onto the same `agent` / `tool` / `llm` span
kinds every other tutorial produces — which is why a CrewAI trace looks at home next to a
gateway trace.

### The direct answer: what you get and what you do not

| | Gateway or SDK path | OTLP path |
|---|---|---|
| Span tree, latency, status | yes | **yes** |
| Model, tokens, dollar cost | yes | **yes** — from the `openai` instrumentor |
| Tool inputs and outputs | yes | **yes** — if payload capture is on (Step 2) |
| Sessions grouping several runs | yes | **yes** — via a `session.id` attribute |
| Prompt versions and lineage | yes | no — the prompt never came from us |
| The tool catalog | yes | no — CrewAI's tools are its own |
| Budgets, rate limits, caching, fallback | yes | no — we are not in the request path |

Nothing on the right-hand side needs code inside the crew. Nothing on the left is reachable from
outside it. That is the trade, and it is per application rather than per call.

### Four traps, all real

**1. Two instrumentors, not one.** `"crewai"` records the orchestration: which agent, which task,
which tool. It does not see the model call, because CrewAI calls the model through the plain
`openai` SDK. List `"openai"` as well or your traces arrive with **zero tokens and no cost**.
Step 9 shows exactly that, with a real run.

**2. `register()` before you import `crewai`.** The instrumentors patch classes when they are
applied. Import the framework first and you are holding references that were never patched. This
notebook keeps the order, and so does every script in the tutorial.

**3. Call `register()` once per process.** Re-run that cell and OpenTelemetry refuses to replace
the global provider, the instrumentors warn that they are already instrumented, and you get back a
provider that is **not** the one exporting your spans. Flushing it then does nothing. In a
notebook this is easy to trip over, because re-running a cell is free.

**4. Nothing is exported until it is flushed.** `BatchSpanProcessor` batches on purpose. A trace
read straight after `kickoff()` can be missing, and a process that exits without flushing loses the
tail of its spans. Step 9 demonstrates both halves.

### The recommendation

Keep the crew ignorant of us. Put `register()` at the top of your entry point, list every layer
that makes calls, and flush before you exit. If you later want prompt versioning or budgets for one
particular call, move that one call to the gateway — the two paths write to the same trace store.

---

## Step 2 — Turn on payload capture

This is the only thing this notebook changes on the platform, and it decides whether the most
interesting part of the trace exists at all: the search query the Researcher chose, and what came
back.

With capture off, you still get the shape — which tool ran, how long it took, whether it failed.
You do not get what went in or out. The setting is per team and applies to every trace, so it is a
deliberate choice about storing model and tool payloads, not a per-run flag.

### In the dashboard

**Observability → Settings.**

| Field | What to set |
|---|---|
| **Capture payloads** | on |

### The same thing in code

**Setup.** Read it first, and only write when it differs — an unnecessary write is still an audit
entry.

In [3]:
settings = await hub.traces.get_settings()
print("capture_payloads is:", settings.capture_payloads)

if not settings.capture_payloads:
    settings = await hub.traces.update_settings(capture_payloads=True)
    print("turned it on:", settings.capture_payloads)
else:
    print("already on, nothing to change")

capture_payloads is: True
already on, nothing to change


---

## Step 3 — Register the OTel pipeline

**Your app.** Three lines, at the top of your entry point, before CrewAI is imported.

`service_name` is what OTel calls the emitting application. `instrument` names the layers to patch.
Keep the returned provider: it is the handle you flush with, and Step 9 shows what happens to a
second one.

Run this cell **once**. If you re-run it, restart the kernel first — trap 3 from Step 1.

In [4]:
from acruxcore.otel import register

tracer_provider = register(
    service_name="crewai-trip-planner",
    instrument=["crewai", "openai"],      # orchestration AND the model call
)
print("provider:", type(tracer_provider).__name__)
print("exporting to: $ACRUXCORE_BASE_URL/traces/otlp")

provider: TracerProvider
exporting to: $ACRUXCORE_BASE_URL/traces/otlp


---

## Step 4 — Build the crew

There is nothing to do in the dashboard for this step. This is ordinary CrewAI, and the point is
that it stays ordinary.

**Your app.** Two agents with different jobs, and one line that decides how work flows between
them: `context=[research_task]` tells the Planner's task to read the Researcher's output.

Note the import order — `crewai` is imported *after* `register()` ran, which is trap 2.

In [5]:
from crewai import Agent, Crew, Process, Task
from crewai_tools import TavilySearchTool

researcher = Agent(
    role="Destination Researcher",
    goal="Find concrete, well-reviewed attractions, restaurants, and neighborhoods for a trip",
    backstory="A travel researcher who always names specific places, never vague categories.",
    tools=[TavilySearchTool(max_results=5)],
    llm=MODEL,
    verbose=False,
)

planner = Agent(
    role="Itinerary Planner",
    goal="Turn research into a clear, day-by-day itinerary",
    backstory="A trip planner who sequences activities sensibly by location and pace.",
    llm=MODEL,
    verbose=False,
)


def build_crew(request: str, prior_itinerary: str = None) -> Crew:
    """One crew: research the request, then write an itinerary from that research."""
    research_task = Task(
        description=f"Research attractions, food, and neighborhoods for: {request}",
        expected_output="A list of specific attractions, restaurants, and neighborhoods.",
        agent=researcher,
    )

    plan_description = "Using the research, write a day-by-day itinerary that satisfies: " + request
    if prior_itinerary:
        # Turn 2 is a real revision: the planner reads turn 1's actual output, not a
        # description of it.
        plan_description = (
            f"Here is the itinerary from the previous turn:\n\n{prior_itinerary}\n\n"
            f"Using the research, revise it to satisfy this new request: {request}")

    plan_task = Task(
        description=plan_description,
        expected_output="A day-by-day itinerary (Day 1, Day 2, Day 3) with specific activities.",
        agent=planner,
        context=[research_task],          # this is the hand-off
    )
    return Crew(agents=[researcher, planner], tasks=[research_task, plan_task],
                process=Process.sequential)


print("agents:", researcher.role, "->", planner.role)
print("researcher tools:", [tool.name for tool in researcher.tools])

agents: Destination Researcher -> Itinerary Planner
researcher tools: ['Tavily Search']


---

## Step 5 — Run turn 1

**Your app.** `using_session` puts a `session.id` attribute on every span emitted inside the block.
That is the only thing tying two separate crew runs together, and it is the reason both turns show
up as one conversation later. CrewAI never learns that the runs are related.

**The trap: `kickoff_async()`, not `kickoff()`.** A Jupyter kernel runs your cells inside an
asyncio event loop, and CrewAI 1.x refuses to start a synchronous run from inside one:

```
RuntimeError: Agent execution was invoked synchronously from within a running event loop.
Use `agent.kickoff_async()` / `crew.kickoff_async()` ...
```

So every cell here awaits `kickoff_async()`. The companion script on GitHub calls plain `kickoff()`
and is right to — a normal Python process has no running loop. The rule is about where you are
calling from, not about which is more correct: inside a notebook, a web handler or any async
framework, use the async entry point.

This takes a minute or two: two agents, a couple of real web searches, four model calls.

In [6]:
from openinference.instrumentation import using_session

with using_session(SESSION_ID):
    crew_1 = build_crew("Plan a 3-day trip to Lisbon focused on food and architecture.")
    result_1 = await crew_1.kickoff_async()

print(str(result_1)[:900])

### 3-Day Itinerary for Lisbon: Food and Architecture

#### Day 1: Exploring Belém's Architectural Marvels and Local Delicacies

**Morning:**
- **Breakfast at Pastéis de Belém**: Start your day with the iconic Pastéis de Belém, known for its delicious custard tarts. Enjoy a coffee while soaking in the ambiance.
  
**Late Morning:**
- **Visit Jerónimos Monastery (Mosteiro dos Jerónimos)**: Explore this stunning example of Gothic architecture and learn about the history of the Age of Discoveries. Don’t miss Vasco da Gama's tomb inside.

**Lunch:**
- **Lunch at Cervejaria Ramiro**: Head to this renowned seafood restaurant for a meal of fresh shellfish and traditional Portuguese dishes.

**Afternoon:**
- **Explore Belém Tower (Torre de Belém)**: After lunch, visit the iconic Belém Tower where you can enjoy panoramic views over the Tagus River. Learn about its historical significance.

**Even


---

## Step 6 — Run turn 2, as a real revision

**Your app.** The same session id, and turn 1's actual text passed back in as `prior_itinerary`.
That distinction matters: a second unrelated plan in the same session would prove nothing about
either the framework or the tracing.

In [7]:
with using_session(SESSION_ID):
    crew_2 = build_crew(
        "Revise the 3-day Lisbon itinerary: make day 2 more relaxed and add one "
        "hands-on cooking class.",
        prior_itinerary=str(result_1),
    )
    result_2 = await crew_2.kickoff_async()

print(str(result_2)[:900])

### Revised 3-Day Lisbon Itinerary with Relaxed Day 2 and a Cooking Class

#### Day 1: Explore Historical Lisbon

**Morning:**
1. **Baixa & Chiado**: Start your morning in the heart of Lisbon by exploring Praça do Comércio, walking along Rua Augusta, and taking in the views from the Santa Justa Lift.

**Late Morning:**
2. **Alfama**: Wander through the cobbled streets of Alfama, visit the iconic **São Jorge Castle**, and enjoy scenic views from **Miradouro de Santa Luzia**.

**Lunch:**
3. **Lunch at Time Out Market**: Sample a variety of local dishes at this vibrant food market, perfect for trying diverse culinary styles.

**Evening:**
4. **Dinner at Farol de Santa Luzia**: Enjoy a traditional meal at this restaurant known for its breathtaking views of the sunset over the river and its authentic Portuguese cuisine.

---

#### Day 2: Relaxed Exploration and Cooking Class

**Morning:**
1. 


---

## Step 7 — Read both runs back

**Check.** Flush first. Spans sit in a batch queue in this process, and a read before the flush can
miss whole traces — Step 9 shows that happening on purpose.

Then look up the session rather than the traces. An OTLP trace does get a name — it takes its root span's,
with the run id trimmed off, so every run of this crew is called `Crew.kickoff`. That is the name of
the *kind* of run, not of this one, so it will not tell two runs apart. `session.id` is what finds
the run you just did.

![The session showing both crew runs, each its own trace, with real token counts and cost](https://docs.acruxcore.com/img/tutorials/trace-a-crewai-trip-planner/02-session.png)

In [8]:
tracer_provider.force_flush()          # export whatever is still queued

found = await hub.traces.list(session_id=SESSION_ID, limit=20)

# The list comes back newest first. Sort it, so "turn 1" means turn 1 even after a re-run.
runs = sorted(found.data, key=lambda t: t.started_at)

print(f"traces in session {SESSION_ID!r}: {found.total}")
for turn, summary in enumerate(runs, start=1):
    print(f"  turn {turn}: {summary.id}  spans={summary.span_count}  "
          f"tokens={summary.total_tokens}  cost=${summary.total_cost_usd}")
    print(f"           name={summary.name!r}")

traces in session 'crewai-trip-planner-notebook': 2
  turn 1: a9c70714-1b01-caad-c18d-a9ed3648a79a  spans=9  tokens=8091  cost=$0.0020691
           name='Crew.kickoff'
  turn 2: 96ad0616-524e-e265-f2a5-dbfdeb919948  spans=10  tokens=9912  cost=$0.00220275
           name='Crew.kickoff'


Two traces per run of this notebook — one per turn — both with real token counts and real dollar
cost, from model calls that never went near our gateway. The `openai` instrumentor read the usage
off OpenAI's own response, and our OTLP endpoint priced it.

Look at the `name` line as well. Both traces are called `Crew.kickoff`, taken from the root span
CrewAI opens for a kickoff. The raw span name is `Crew_<uuid>.kickoff`; the run id is trimmed so
that two runs of the same crew share one searchable name rather than each getting a unique one.
Shared is the point — and it is also why this step still looks the run up by session and sorts by
start time, because the name cannot tell turn 1 from turn 2.

**Check.** Now the span tree. This is CrewAI's own execution order, captured with no tracing code
in the crew.

![The trace tree: a chain root span, an agent span containing an LLM span and two search tool spans, then the Planner agent's span](https://docs.acruxcore.com/img/tutorials/trace-a-crewai-trip-planner/03-trace-tree.png)

In [9]:
def walk(spans, depth=0):
    """Print the span tree. A tool span is a child of the model turn that asked for it."""
    for span in spans:
        label = (span.model or span.name or "")[:44]
        cost = f" ${span.cost_usd}" if span.cost_usd else ""
        tokens = f" {span.total_tokens}tok" if span.total_tokens else ""
        print("  " * (depth + 1) + f"{span.kind:<9} {label}{tokens}{cost}")
        walk(span.children, depth + 1)


first = await hub.traces.get(runs[0].id)        # runs[] is sorted oldest first
print("turn 1 tree:")
walk(first.spans)

turn 1 tree:
  chain     Crew_0547574c-217d-4d43-8650-337fcb278632.ki
    agent     Destination Researcher._execute_core
      llm       gpt-4o-mini-2024-07-18 244tok $6.99e-05
      tool      Tavily Search.run
      tool      Tavily Search.run
      tool      Tavily Search.run
      llm       gpt-4o-mini-2024-07-18 5884tok $0.0013443
    agent     Itinerary Planner._execute_core
      llm       gpt-4o-mini-2024-07-18 1963tok $0.0006549


The `kind` column is the interesting part. CrewAI emitted OpenInference attributes; our endpoint
turned them into `chain`, `agent`, `tool` and `llm` — the same four kinds a hand-written
`run_tool_loop` trace uses. Nothing about this trace is second-class because it arrived over OTLP.

**The span names are not ours and not stable.** `Crew_<uuid>` and `Haiku Writer._execute_core` come
from `openinference-instrumentation-crewai`, and they changed shape between CrewAI 0.x and 1.x. The
companion page's screenshot was taken on an earlier version and shows `Crew.kickoff` at the root.
Read span names, do not build on them.

**Check.** The tool span, which is where payload capture earns its keep: the query the Researcher
chose for itself, and what actually came back.

![The expanded search tool span showing the real query and real search results](https://docs.acruxcore.com/img/tutorials/trace-a-crewai-trip-planner/04-tool-span.png)

In [10]:
def every_span(spans):
    for span in spans:
        yield span
        yield from every_span(span.children)


tool_spans = [s for s in every_span(first.spans) if s.kind == "tool"]
print(f"{len(tool_spans)} tool span(s) in turn 1\n")

for span in tool_spans:
    payload = span.payload or {}
    print(f"{span.name}  ({span.latency_ms} ms)")
    print(f"  input:  {json.dumps(payload.get('input'))[:200]}")
    print(f"  output: {json.dumps(payload.get('output'))[:260]}")
    print()

3 tool span(s) in turn 1

Tavily Search.run  (1606 ms)
  input:  "{\"query\": \"Lisbon attractions architecture\"}"
  output: "{\n  \"query\": \"Lisbon attractions architecture\",\n  \"follow_up_questions\": null,\n  \"answer\": null,\n  \"images\": [],\n  \"results\": [\n    {\n      \"url\": \"https://theforeignarchitect.com/guides/lisbon\",\n      \"title\": \"Lisbon Contemporary 

Tavily Search.run  (1919 ms)
  input:  "{\"query\": \"Lisbon food restaurants\"}"
  output: "{\n  \"query\": \"Lisbon food restaurants\",\n  \"follow_up_questions\": null,\n  \"answer\": null,\n  \"images\": [],\n  \"results\": [\n    {\n      \"url\": \"https://restlessfeet.com/lisbon-restaurant-scene-portugal\",\n      \"title\": \"Lisbon Restauran

Tavily Search.run  (1849 ms)
  input:  "{\"query\": \"Lisbon neighborhoods hot spots\"}"
  output: "{\n  \"query\": \"Lisbon neighborhoods hot spots\",\n  \"follow_up_questions\": null,\n  \"answer\": null,\n  \"images\": [],\n  \"results\": [\n    {\n      

---

## Step 8 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code. Each one runs a tiny
one-agent crew so it stays cheap.

### Mistake 1 — instrumenting only `crewai`

**Broken on purpose.** The single most common way to end up with a trace that looks fine and tells
you nothing about cost. `crewai` sees the orchestration; the model call happens one layer down, in
the `openai` SDK.

This cell removes the `openai` instrumentation, runs a crew, and puts it back.

In [11]:
from openinference.instrumentation.openai import OpenAIInstrumentor

tiny_agent = Agent(role="Counter", goal="Count", backstory="A counter.", llm=MODEL, verbose=False)


def tiny_crew() -> Crew:
    task = Task(description="Say the word two.", expected_output="One word.", agent=tiny_agent)
    return Crew(agents=[tiny_agent], tasks=[task], process=Process.sequential)


OpenAIInstrumentor().uninstrument()          # broken on purpose: only crewai is left
with using_session("crewai-notebook-no-openai"):
    await tiny_crew().kickoff_async()
tracer_provider.force_flush()

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)   # put it back
print("openai instrumentor restored\n")

broken = await hub.traces.list(session_id="crewai-notebook-no-openai", limit=5)
for summary in broken.data:
    print(f"spans={summary.span_count}  tokens={summary.total_tokens}  "
          f"cost={summary.total_cost_usd}")
    detail = await hub.traces.get(summary.id)
    print("kinds present:", sorted({s.kind for s in every_span(detail.spans)}))

openai instrumentor restored

spans=2  tokens=0  cost=None
kinds present: ['agent', 'chain']


No `llm` span, no tokens, no cost — and no error anywhere to tell you. Compare that with the three
kinds in Step 7.

### Mistake 2 — reading the trace before flushing

**Broken on purpose.** `BatchSpanProcessor` holds spans and exports them in batches, which is what
makes OTel cheap enough to leave on. The cost is that a read immediately after a run can find
nothing at all.

If you have run this notebook before, both counts below will be higher than on a first run. What
matters is that the second one is larger than the first.

In [12]:
with using_session("crewai-notebook-flush-timing"):
    await tiny_crew().kickoff_async()

before = await hub.traces.list(session_id="crewai-notebook-flush-timing", limit=5)
print(f"before flush: {before.total} trace(s)")

tracer_provider.force_flush()

after = await hub.traces.list(session_id="crewai-notebook-flush-timing", limit=5)
print(f"after flush:  {after.total} trace(s)")
for summary in after.data:
    print(f"  spans={summary.span_count}  tokens={summary.total_tokens}")

before flush: 0 trace(s)
after flush:  1 trace(s)
  spans=3  tokens=68


The same lesson bites harder in a script than in a notebook: a process that exits without flushing
drops whatever was still queued, and the spans it loses are the ones at the end of the run — the
part you were most likely watching for.

### Mistake 3 — forgetting `using_session`

**Broken on purpose.** Without the context manager the run is still traced perfectly. It just has
no `session.id`, so nothing connects it to the turn before it, and the session view will never show
it.

In [13]:
await tiny_crew().kickoff_async()      # broken on purpose: no using_session(...) around it
tracer_provider.force_flush()

recent = await hub.traces.list(limit=5)
for summary in recent.data[:3]:
    print(f"{summary.id}  session={summary.session_id!r}  spans={summary.span_count}")
print("\nA trace with session=None is invisible to Observability -> Sessions.")

7bbd1209-e32f-5444-37fc-e9a4e3c51598  session=None  spans=3
cd6c4c2b-7a81-d699-43ff-b84da9ae082f  session='crewai-notebook-flush-timing'  spans=3
1ff9b11f-4ef8-b788-ab36-91d36f1ee62c  session='crewai-notebook-no-openai'  spans=2

A trace with session=None is invisible to Observability -> Sessions.


### Mistake 4 — calling `register()` twice

**Broken on purpose.** The one that is specific to notebooks, because re-running a cell costs
nothing. OpenTelemetry allows exactly one global `TracerProvider` per process. A second
`register()` cannot replace it, so it hands back a provider that is **not** wired to your exporter.

Watch the two log lines this cell produces, and then the identity check.

In [14]:
second = register(service_name="crewai-trip-planner", instrument=["crewai", "openai"])

print("is it the same provider we have been flushing?", second is tracer_provider)
print("the instrumentors and every span still go to the FIRST provider.")
second.force_flush()      # returns True, and exports nothing: its queue is always empty
print()
print("Fix: call register() once. To change it in a notebook, restart the kernel -")
print("keeping a stale provider is worse than the duplicate warnings suggest.")

is it the same provider we have been flushing? False
the instrumentors and every span still go to the FIRST provider.

Fix: call register() once. To change it in a notebook, restart the kernel -
keeping a stale provider is worse than the duplicate warnings suggest.


---

## Step 9 — Flush before you finish

**Your app.** The last thing your entry point does. In a script this goes at the end of `main()`;
here it is a cell.

`hub` is only used by this notebook's **Check** cells, but it holds an HTTP pool, so close it too.

In [15]:
tracer_provider.force_flush()
await hub.gateway.aclose()
print("flushed")

flushed


---

## What you built

A two-agent crew that plans a trip and then revises it, fully traced — agents, tools, model calls,
tokens and cost — with the crew itself knowing nothing about AcruxCore.

### What of this actually ships

Three lines above your crew, and one at the end:

```python
import os
from acruxcore.otel import register

os.environ["CREWAI_TRACING_ENABLED"] = "false"      # CrewAI's own service, not ours

tracer_provider = register(
    service_name="crewai-trip-planner",
    instrument=["crewai", "openai"],                 # orchestration AND the model call
)

from crewai import Agent, Crew, Process, Task        # imported AFTER register()
from crewai_tools import TavilySearchTool
from openinference.instrumentation import using_session

# ... your crew, exactly as you already wrote it ...

def main() -> None:
    with using_session("crewai-trip-planner-demo"):
        result_1 = build_crew("Plan a 3-day trip to Lisbon...").kickoff()

    with using_session("crewai-trip-planner-demo"):
        result_2 = build_crew("Revise it: ...", prior_itinerary=str(result_1)).kickoff()

    tracer_provider.force_flush()                    # do not skip this
```

Note `kickoff()` there and `await kickoff_async()` in the cells above. A script has no running
event loop, so the synchronous entry point is correct; a notebook, a web handler or anything
already inside asyncio must use the async one. Same crew either way.

That is the entire integration. Everything else in this notebook was scaffolding:

- the preflight and every **Check** cell read state back to prove a step worked. None of it belongs
  in a request path.
- Step 2 is a one-time team setting, and the dashboard does the same job.
- Step 8 is all deliberately broken, and it leaves three extra traces behind.

### The four rules worth remembering

1. **Two instrumentors.** `crewai` alone gives you a trace with no tokens and no cost.
2. **`register()` first, then import the framework.**
3. **`register()` once per process.** A second call returns a provider that flushes nothing.
4. **Flush before you exit**, or lose the tail of every run.
5. **`kickoff_async()` inside an event loop**, `kickoff()` in a plain script.

### What this notebook left in your team

- payload capture on, if it was not already
- five traces: two real crew runs grouped in one session, plus three from Step 8
- nothing else — no prompts, no tools, no models. That is what "no AcruxCore code in the crew"
  means in practice.

### Where to go next

- [Trace an OpenAI Agents SDK support-triage system](https://docs.acruxcore.com/docs/tutorials/trace-an-openai-agents-sdk-triage-system)
  — the same OTLP path on a framework whose agent-to-agent handoffs make a differently shaped tree.
- [Send OTel traces to AcruxCore with the SDK helper](https://docs.acruxcore.com/docs/guides/send-otel-traces-with-the-sdk-helper)
  — `register()` on its own: a bare pipeline, one framework, several at once.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — what else a session gives you once your runs are grouped.